# Contract Intelligence — Clause Classifier Training

Trains **Legal-BERT** and **DeBERTa-v3** multi-label clause classifiers, runs model selection, and packages checkpoints for download.

---

### Before running
1. **Enable GPU**: `Runtime → Change runtime type → T4 GPU`
2. **Upload data to Drive** — copy these 3 files from your local machine into Drive:
```
My Drive/contract-intelligence-platform/data/processed/train.csv
My Drive/contract-intelligence-platform/data/processed/val.csv
My Drive/contract-intelligence-platform/data/processed/test.csv
```

### After running — place downloads locally at:
```
contract-intelligence-platform/
└── models/
    └── clause_classifier/
        ├── legalbert/best/        ← unzip legalbert_best.zip here
        ├── deberta/best/          ← unzip deberta_best.zip here
        └── production_config.json
```

## Step 1 — GPU Check, Drive Mount & Dependencies

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected.')
    print('Go to Runtime -> Change runtime type -> T4 GPU before continuing.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'sentencepiece', 'protobuf',
    'tokenizers', 'pandas', 'numpy', 'scikit-learn', 'tqdm', 'loguru', 'torch'
])
print('Dependencies installed.')

## Step 2 — Configuration & Path Setup

In [ ]:
import json
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT_DIR      = Path('/content/drive/MyDrive/contract-intelligence-platform')
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
LOGS_DIR      = ROOT_DIR / 'logs'

CLASSIFIER_DIR           = ROOT_DIR / 'models' / 'clause_classifier'
CLASSIFIER_LEGALBERT_DIR = CLASSIFIER_DIR / 'legalbert'
CLASSIFIER_DEBERTA_DIR      = CLASSIFIER_DIR / 'deberta'
CLASSIFIER_LEGALROBERTA_DIR = CLASSIFIER_DIR / 'legalroberta'
PRODUCTION_CONFIG_PATH   = CLASSIFIER_DIR / 'production_config.json'

# ── HuggingFace model identifiers ─────────────────────────────────────────
LEGAL_BERT_MODEL = 'nlpaueb/legal-bert-base-uncased'
DEBERTA_MODEL         = 'microsoft/deberta-v3-base'
LEGAL_ROBERTA_MODEL   = 'lexlms/legal-roberta-large'

# ── Unified clause taxonomy (100 base + dynamic EDGAR types) ──────────────
CUAD_CLAUSE_TYPES = [
    # CUAD core (41)
    'Competitive Restriction Exception', 'Non-Compete', 'Exclusivity',
    'No-Solicit Of Customers', 'No-Solicit Of Employees', 'Non-Disparagement',
    'Termination For Convenience', 'Rofr/Rofo/Rofn', 'Change Of Control',
    'Anti-Assignment', 'Revenue/Profit Sharing', 'Price Restrictions',
    'Minimum Commitment', 'Volume Restriction', 'IP Ownership Assignment',
    'Joint IP Ownership', 'License Grant', 'Non-Transferable License',
    'Affiliate License-Licensor', 'Affiliate License-Licensee',
    'Unlimited/All-You-Can-Eat-License', 'Irrevocable Or Perpetual License',
    'Source Code Escrow', 'Post-Termination Services', 'Audit Rights',
    'Uncapped Liability', 'Cap On Liability', 'Liquidated Damages',
    'Warranty Duration', 'Insurance', 'Covenant Not To Sue',
    'Third Party Beneficiary', 'Most Favored Nation', 'Governing Law',
    'Dispute Resolution', 'Limitations Of Liability', 'Indemnification',
    'Agreement Date', 'Effective Date', 'Expiration Date', 'Renewal Term',
    # New from LEDGAR (32)
    'Confidentiality', 'Amendments', 'Anti-Corruption Laws',
    'Approvals And Consents', 'Authority', 'Compliance With Laws',
    'Consent To Jurisdiction', 'Definitions', 'Enforceability',
    'Entire Agreements', 'Expenses', 'Force Majeure', 'Further Assurances',
    'Interests', 'Liens And Encumbrances', 'Limitations Of Remedies',
    'Non-Waiver', 'Notices', 'Organization And Existence', 'Payments',
    'Representations And Warranties', 'Sanctions', 'Securities Law Compliance',
    'Severability', 'Specific Performance', 'Successors And Assigns',
    'Survival', 'Taxes And Withholding', 'Trade Controls',
    'Transactions With Affiliates', 'Waiver Of Jury Trial', 'Waivers',
    # New from MAUD (14)
    'No-Shop', 'Material Adverse Effect', 'Hell-Or-High-Water',
    'Termination Fee', 'Reverse Termination Fee', 'Matching Rights',
    'Superior Offer Definition', 'Fiduciary Exception',
    'Antitrust Efforts Standard', 'Operating Covenants',
    'Expense Reimbursement', 'Intervening Event Definition',
    'Board Recommendation Change', 'Tail Period',
    # Additional commercial provisions (13)
    'Data Protection And Privacy', 'Employment And Benefits', 'Capitalization',
    'Publicity And Announcements', 'Non-Solicitation General', 'Step-In Rights',
    'Benchmarking Rights', 'Electronic Signatures And Counterparts',
    'Service Level Agreement', 'Subcontracting', 'Assignment Of Receivables',
    'Performance Bonds', 'Escrow And Holdback',
]
# ── Dynamic taxonomy ───────────────────────────────────────────────────────
DYNAMIC_TAXONOMY_PATH = PROCESSED_DIR / 'dynamic_taxonomy.json'
if DYNAMIC_TAXONOMY_PATH.exists():
    _dynamic = json.load(open(DYNAMIC_TAXONOMY_PATH))
    CUAD_CLAUSE_TYPES = CUAD_CLAUSE_TYPES + [t for t in _dynamic if t not in CUAD_CLAUSE_TYPES]
    print(f'Dynamic taxonomy loaded: {len(_dynamic)} types merged.')
else:
    print('WARNING: dynamic_taxonomy.json not found.')

NUM_CLAUSE_TYPES = len(CUAD_CLAUSE_TYPES)

# ── Training hyperparameters ───────────────────────────────────────────────
LEARNING_RATE = {'legalbert': 2e-5,  'deberta': 3e-6}
EPOCHS        = {'legalbert': 90,    'deberta': 90}
HEAD_LR_MULT  = {'legalbert': 10,    'deberta': 1}
ADAM_EPS      = {'legalbert': 1e-8,  'deberta': 1e-6}
GRAD_CLIP_NORM = {'legalbert': 1.0,  'deberta': 0.5}

MIN_EPOCHS_BEFORE_STOPPING = {'legalbert': 10, 'deberta': 20}
EARLY_STOPPING_PATIENCE    = {'legalbert': 6,  'deberta': 8, 'legalroberta': 6}

BACKBONE_USE_MEAN_POOL = {'legalbert': False, 'deberta': True, 'legalroberta': False}

BATCH_SIZE           = 16
BACKBONE_BATCH_SIZE  = {'legalbert': BATCH_SIZE, 'deberta': BATCH_SIZE, 'legalroberta': BATCH_SIZE // 2}
BACKBONE_GRAD_ACCUM  = {'legalbert': 2, 'deberta': 2, 'legalroberta': 4}
WARMUP_STEPS         = 500
# 256 covers 95%+ of legal clauses; 512→256 gives ~4× speedup in attention
MAX_LENGTH           = 256
UNKNOWN_THRESH       = 0.3
CLASSIFIER_THRESHOLD = 0.5

BACKBONE_MODEL_MAP = {'legalbert': LEGAL_BERT_MODEL, 'deberta': DEBERTA_MODEL, 'legalroberta': LEGAL_ROBERTA_MODEL}
BACKBONE_DIR_MAP   = {'legalbert': CLASSIFIER_LEGALBERT_DIR, 'deberta': CLASSIFIER_DEBERTA_DIR, 'legalroberta': CLASSIFIER_LEGALROBERTA_DIR}

for d in [PROCESSED_DIR, LOGS_DIR, CLASSIFIER_LEGALBERT_DIR, CLASSIFIER_DEBERTA_DIR, CLASSIFIER_LEGALROBERTA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

missing = [f for f in ['train.csv', 'val.csv', 'test.csv'] if not (PROCESSED_DIR / f).exists()]
if missing:
    print(f'MISSING FILES: {missing}')
else:
    import pandas as pd
    for name in ['train.csv', 'val.csv', 'test.csv']:
        df = pd.read_csv(PROCESSED_DIR / name)
        print(f'{name}: {len(df):,} rows')
    print(f'\nClause types: {NUM_CLAUSE_TYPES}')
    print('All data files found. Ready to train.')

## Step 3 — Imports & Utilities

In [ ]:
import json
import math
import sys
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from loguru import logger
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

logger.remove()
logger.add(str(LOGS_DIR / 'classifier.log'), rotation='10 MB', level='DEBUG')
logger.add(sys.stderr, level='INFO')
print('Imports complete.')

In [ ]:
# ── Data loading & label encoding ─────────────────────────────────────────

def load_split(split_name: str) -> Tuple[List[str], List[List[int]]]:
    '''Load a CSV split and return (texts, multi-hot labels).'''
    path = PROCESSED_DIR / f'{split_name}.csv'
    if not path.exists():
        raise FileNotFoundError(f'{path} not found. Upload it to Drive first.')
    df     = pd.read_csv(path)
    texts  = df['clause_text'].fillna('').tolist()
    labels = [encode_labels(r) for r in df['clause_type'].fillna('').tolist()]
    return texts, labels


def encode_labels(clause_type_str: str) -> List[int]:
    '''Convert pipe-separated clause type string to multi-hot vector.'''
    vec = [0] * NUM_CLAUSE_TYPES
    if not clause_type_str.strip():
        return vec
    idx_map = {ct: i for i, ct in enumerate(CUAD_CLAUSE_TYPES)}
    for ct in clause_type_str.split('|'):
        ct = ct.strip()
        if ct in idx_map:
            vec[idx_map[ct]] = 1
    return vec


def build_param_groups(hf_model, lr: float, head_mult: int, no_decay):
    '''Backbone-agnostic optimizer param groups with layer-wise LR.

    Works for both ClauseClassifierModel (head at .classifier / .pooler)
    and MeanPoolClassifierModel (head at .classifier directly on the object).
    '''
    all_named = list(hf_model.named_parameters())
    head_ids  = set()
    for attr in ('classifier', 'pooler'):
        sub = getattr(hf_model, attr, None)
        if sub is not None:
            for p in sub.parameters():
                head_ids.add(id(p))
    return [
        {'params': [p for n, p in all_named if id(p) not in head_ids and not any(nd in n for nd in no_decay)], 'lr': lr,             'weight_decay': 0.01},
        {'params': [p for n, p in all_named if id(p) not in head_ids and     any(nd in n for nd in no_decay)], 'lr': lr,             'weight_decay': 0.0},
        {'params': [p for n, p in all_named if id(p) in head_ids],                                             'lr': lr * head_mult, 'weight_decay': 0.01},
    ]

print('Data utilities defined.')

## Step 4 — Model & Dataset Classes

In [ ]:
class ClauseDataset(Dataset):
    '''Pre-tokenization + sliding-window chunking.

    Short clauses (≤ max_length tokens): batch-tokenized upfront — zero CPU
    work per batch during training.

    Long clauses (> max_length tokens): split into overlapping chunks of
    max_length tokens (stride = max_length // 2). Each chunk inherits the
    parent clause labels so the model sees the full text including exceptions,
    limitations, and conditions that often appear beyond the first max_length
    tokens.
    '''
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH, stride=128):
        logger.info(f'Pre-tokenizing {len(texts):,} texts (runs once)...')

        cls_id    = tokenizer.cls_token_id
        sep_id    = tokenizer.sep_token_id
        pad_id    = tokenizer.pad_token_id
        max_chunk = max_length - 2  # reserve for [CLS] and [SEP]

        # Step 1: tokenize all texts without padding to get lengths
        raw = tokenizer(
            texts,
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_attention_mask=False,
        )
        raw_ids = raw['input_ids']

        # Step 2: partition into short and long
        short_texts, short_labels = [], []
        long_ids_list, long_labels_list = [], []

        for i, ids in enumerate(raw_ids):
            if len(ids) <= max_length:
                short_texts.append(texts[i])
                short_labels.append(labels[i])
            else:
                long_ids_list.append(ids)
                long_labels_list.append(labels[i])

        parts_ids, parts_masks, parts_lbls = [], [], []

        # Step 3: batch-tokenize short texts (fast)
        if short_texts:
            enc = tokenizer(
                short_texts,
                max_length=max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt',
            )
            parts_ids.append(enc['input_ids'])
            parts_masks.append(enc['attention_mask'])
            parts_lbls.append(torch.tensor(short_labels, dtype=torch.float32))

        # Step 4: sliding-window chunk long texts
        chunk_ids, chunk_masks, chunk_lbls = [], [], []
        n_long_chunks = 0

        for ids, label in zip(long_ids_list, long_labels_list):
            body  = ids[1:-1]  # strip [CLS] and [SEP]
            start = 0
            while start < len(body):
                chunk = body[start : start + max_chunk]
                seq   = [cls_id] + chunk + [sep_id]
                pad_n = max_length - len(seq)
                chunk_ids.append(seq + [pad_id] * pad_n)
                chunk_masks.append([1] * len(seq) + [0] * pad_n)
                chunk_lbls.append(label)
                n_long_chunks += 1
                next_start = start + max_chunk - stride
                if next_start >= len(body):
                    break
                start = next_start

        if chunk_ids:
            parts_ids.append(torch.tensor(chunk_ids,   dtype=torch.long))
            parts_masks.append(torch.tensor(chunk_masks, dtype=torch.long))
            parts_lbls.append(torch.tensor(chunk_lbls,  dtype=torch.float32))

        self.input_ids      = torch.cat(parts_ids,   dim=0)
        self.attention_mask = torch.cat(parts_masks, dim=0)
        self.labels         = torch.cat(parts_lbls,  dim=0)

        logger.info(
            f'Pre-tokenization complete. '
            f'{len(texts):,} clauses → {len(self.labels):,} training examples '
            f'({len(long_ids_list):,} long clauses expanded into '
            f'{n_long_chunks:,} chunks via sliding window, stride={stride}).'
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels':         self.labels[idx],
        }


class ClauseClassifierModel(nn.Module):
    '''Standard backbone classifier — used for Legal-BERT.'''
    def __init__(self, model_name, num_labels=NUM_CLAUSE_TYPES):
        super().__init__()
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            problem_type='multi_label_classification',
            ignore_mismatched_sizes=True,
        )

    def forward(self, input_ids, attention_mask):
        return self.model(input_ids=input_ids, attention_mask=attention_mask).logits


class MeanPoolClassifierModel(nn.Module):
    '''Mean-pooling classifier — used for DeBERTa-v3.'''
    def __init__(self, model_name, num_labels=NUM_CLAUSE_TYPES):
        super().__init__()
        cfg             = AutoConfig.from_pretrained(model_name, ignore_mismatched_sizes=True)
        self.encoder    = AutoModel.from_pretrained(model_name, config=cfg)
        hidden          = cfg.hidden_size
        self.norm       = nn.LayerNorm(hidden)
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, num_labels)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask):
        outputs  = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden   = outputs.last_hidden_state
        mask_exp = attention_mask.unsqueeze(-1).float()
        pooled   = (hidden * mask_exp).sum(dim=1) / mask_exp.sum(dim=1).clamp(min=1e-9)
        return self.classifier(self.dropout(self.norm(pooled)))

    def save_pretrained(self, save_dir):
        import os
        os.makedirs(save_dir, exist_ok=True)
        self.encoder.save_pretrained(save_dir)
        torch.save(
            {'norm': self.norm.state_dict(), 'classifier': self.classifier.state_dict()},
            os.path.join(save_dir, 'head_weights.pt'),
        )

    @classmethod
    def from_pretrained(cls, save_dir, num_labels=NUM_CLAUSE_TYPES):
        import os
        model = cls(save_dir, num_labels=num_labels)
        head_path = os.path.join(save_dir, 'head_weights.pt')
        if os.path.exists(head_path):
            state = torch.load(head_path, map_location='cpu')
            model.norm.load_state_dict(state['norm'])
            model.classifier.load_state_dict(state['classifier'])
        return model


def build_model(backbone, model_name):
    if BACKBONE_USE_MEAN_POOL.get(backbone, False):
        return MeanPoolClassifierModel(model_name)
    return ClauseClassifierModel(model_name)


class AsymmetricLoss(nn.Module):
    '''ASL — Ben-Baruch et al. (ICCV 2021).'''
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, reduction='mean'):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip      = clip
        self.reduction = reduction

    def forward(self, logits, targets):
        probs   = torch.sigmoid(logits)
        probs_m = (probs + self.clip).clamp(max=1) if self.clip > 0 else probs
        loss_pos = targets       * torch.log(probs.clamp(min=1e-8))
        loss_neg = (1 - targets) * torch.log((1 - probs_m).clamp(min=1e-8))
        with torch.no_grad():
            w_p = torch.pow(1 - probs.detach(),  self.gamma_pos)
            w_n = torch.pow(probs_m.detach(),     self.gamma_neg)
            w   = w_p * targets + w_n * (1 - targets)
        loss = -w * (loss_pos + loss_neg)
        return loss.mean() if self.reduction == 'mean' else loss.sum()

print('Classes defined: ClauseDataset (pre-tokenize + sliding window), ClauseClassifierModel, MeanPoolClassifierModel, AsymmetricLoss, build_model.')

## Step 5 — Trainer & Model Selector

In [ ]:
GRADIENT_ACCUM_STEPS = BACKBONE_GRAD_ACCUM.get(backbone, 2)
bs = BACKBONE_BATCH_SIZE.get(backbone, BATCH_SIZE)

def _evaluate_aggregated(model, tokenizer, texts, labels, device, max_length=MAX_LENGTH, batch_size=64):
    """Evaluate on raw clause texts with sliding-window + max-pool aggregation.

    Each clause produces exactly ONE prediction regardless of length.
    Long clauses are split into overlapping chunks, run through the model,
    and max-pooled back into one prediction per original clause.
    Matches the inference path exactly — F1 is per original clause, not per chunk.
    """
    use_amp   = (device == 'cuda')
    stride    = 128
    max_chunk = max_length - 2
    cls_id    = tokenizer.cls_token_id
    sep_id    = tokenizer.sep_token_id
    pad_id    = tokenizer.pad_token_id

    all_preds, all_labels = [], []
    model.eval()

    with torch.no_grad():
        for text, label in tqdm(zip(texts, labels), total=len(texts), desc='  Evaluating', leave=False):
            token_ids = tokenizer.encode(text, add_special_tokens=False)

            if len(token_ids) <= max_length - 2:
                # Short clause — single forward pass
                enc = tokenizer(
                    text, max_length=max_length, padding='max_length',
                    truncation=True, return_tensors='pt',
                ).to(device)
                with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                    logits = model(enc['input_ids'], enc['attention_mask'])
                probs = torch.sigmoid(logits).float().cpu().numpy()[0]
            else:
                # Long clause — sliding window, max-pool across chunks
                chunks_ids, chunks_mask = [], []
                start = 0
                while start < len(token_ids):
                    chunk = token_ids[start : start + max_chunk]
                    seq   = [cls_id] + chunk + [sep_id]
                    pad_n = max_length - len(seq)
                    chunks_ids.append(seq + [pad_id] * pad_n)
                    chunks_mask.append([1] * len(seq) + [0] * pad_n)
                    next_start = start + max_chunk - stride
                    if next_start >= len(token_ids):
                        break
                    start = next_start

                chunk_probs = []
                for i in range(0, len(chunks_ids), batch_size):
                    ids_t  = torch.tensor(chunks_ids[i:i+batch_size],  dtype=torch.long).to(device)
                    mask_t = torch.tensor(chunks_mask[i:i+batch_size], dtype=torch.long).to(device)
                    with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                        logits = model(ids_t, mask_t)
                    chunk_probs.append(torch.sigmoid(logits).float().cpu().numpy())

                probs = np.max(np.vstack(chunk_probs), axis=0)  # max-pool across chunks

            all_preds.append((probs >= 0.5).astype(int))
            all_labels.append(label)

    y_pred = np.vstack(all_preds)
    y_true = np.array(all_labels)
    return {
        'f1':        f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall':    recall_score(y_true, y_pred, average='macro', zero_division=0),
    }


def train_backbone(backbone: str) -> Dict:
    """
    Train one backbone end-to-end with fp16 mixed precision and auto-resume.

    Training:   ClauseDataset chunks long clauses into overlapping windows.
                Each chunk is an independent training example.
    Validation: _evaluate_aggregated() — sliding window + max-pool per clause.
                One prediction per original clause, matching the inference path.

    Saves after every epoch:
      - model weights + tokenizer  → ckpt_dir/best/   (on F1 improvement only)
      - optimizer + scheduler state → ckpt_dir/training_state.pt
      - training metadata           → ckpt_dir/training_state.json
    """
    model_name = BACKBONE_MODEL_MAP[backbone]
    ckpt_dir   = BACKBONE_DIR_MAP[backbone]
    lr         = LEARNING_RATE[backbone]
    epochs     = EPOCHS[backbone]
    head_mult  = HEAD_LR_MULT[backbone]
    adam_eps   = ADAM_EPS[backbone]
    clip_norm  = GRAD_CLIP_NORM[backbone]
    device     = 'cuda' if torch.cuda.is_available() else 'cpu'
    use_fast   = (backbone != 'legalbert')
    use_amp    = (device == 'cuda')

    state_json = ckpt_dir / 'training_state.json'
    state_pt   = ckpt_dir / 'training_state.pt'
    best_dir   = ckpt_dir / 'best'
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # -- Auto-detect resume ------------------------------------------------
    resuming = state_json.exists() and best_dir.exists()
    if resuming:
        saved       = json.loads(state_json.read_text())
        start_epoch = saved['epoch'] + 1
        best_f1     = saved['best_f1']
        best_epoch  = saved['best_epoch']
        no_improve  = saved['no_improve']
        history     = saved['history']
        logger.info(f'[{backbone}] RESUMING from epoch {start_epoch} '
                    f'(best F1={best_f1:.4f} at epoch {best_epoch})')
        tokenizer = AutoTokenizer.from_pretrained(str(best_dir), use_fast=use_fast)
        if BACKBONE_USE_MEAN_POOL.get(backbone, False):
            model = MeanPoolClassifierModel.from_pretrained(str(best_dir)).to(device)
        else:
            model = ClauseClassifierModel(str(best_dir)).to(device)
    else:
        start_epoch = 1
        best_f1, best_epoch, no_improve = 0.0, 0, 0
        history     = []
        logger.info(f'[{backbone}] Fresh run -- lr={lr}  epochs={epochs}  device={device}  fp16={use_amp}')
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=use_fast)
        model     = build_model(backbone, model_name).to(device)
    if backbone == 'legalroberta':
        if hasattr(model, 'model'):
            model.model.gradient_checkpointing_enable()
        elif hasattr(model, 'encoder'):
            model.encoder.gradient_checkpointing_enable()

    train_texts, train_labels = load_split('train')
    val_texts,   val_labels   = load_split('val')

    # Training loader: ClauseDataset chunks long clauses (independent examples)
    train_loader = DataLoader(
        ClauseDataset(train_texts, train_labels, tokenizer),
        batch_size=bs, shuffle=True, num_workers=2,
        pin_memory=(device == 'cuda'),
    )
    # Val: raw texts + labels kept for aggregated evaluation (one pred per clause)

    criterion    = AsymmetricLoss(gamma_neg=4.0, gamma_pos=1.0, clip=0.05)
    scaler       = torch.amp.GradScaler(device='cuda', enabled=use_amp)

    hf_model_for_params = (
        model if BACKBONE_USE_MEAN_POOL.get(backbone, False) else model.model
    )
    optimizer = torch.optim.AdamW(
        build_param_groups(hf_model_for_params, lr, head_mult, ['bias', 'LayerNorm.weight']),
        eps=adam_eps,
    )
    total_steps = (len(train_loader) // GRADIENT_ACCUM_STEPS) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps
    )

    if resuming and state_pt.exists():
        pt = torch.load(str(state_pt), map_location=device)
        optimizer.load_state_dict(pt['optimizer'])
        scheduler.load_state_dict(pt['scheduler'])
        logger.info(f'[{backbone}] Optimizer and scheduler state restored.')

    patience   = EARLY_STOPPING_PATIENCE[backbone]
    min_epochs = MIN_EPOCHS_BEFORE_STOPPING[backbone]

    for epoch in range(start_epoch, epochs + 1):
        # -- Train ---------------------------------------------------------
        model.train()
        total_loss, nan_batches = 0.0, 0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs} [{backbone}]', leave=True)

        for step, batch in enumerate(pbar):
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)

            with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                logits = model(ids, mask)

            if torch.isnan(logits).any():
                nan_batches += 1
                optimizer.zero_grad()
                if nan_batches > 50:
                    logger.error(f'[{backbone}] >50 NaN batches -- aborting.')
                    avg_loss = float('nan')
                    break
                continue

            with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                loss = criterion(logits, lbls) / GRADIENT_ACCUM_STEPS
            scaler.scale(loss).backward()
            total_loss += loss.item() * GRADIENT_ACCUM_STEPS

            if (step + 1) % GRADIENT_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            pbar.set_postfix({'loss': f'{loss.item() * GRADIENT_ACCUM_STEPS:.4f}'})

        avg_loss = total_loss / max(len(train_loader) - nan_batches, 1)
        if nan_batches:
            logger.warning(f'Epoch {epoch}: skipped {nan_batches} NaN batches')

        # -- NaN guard: skip val if epoch was aborted --
        if math.isnan(avg_loss):
            print(f'  Epoch {epoch} aborted (NaN) — skipping val eval, no_improve unchanged.')
            history.append({'epoch': epoch, 'train_loss': avg_loss,
                             'f1_macro': float('nan'), 'precision_macro': float('nan'),
                             'recall_macro': float('nan')})
            continue

        # -- Validate (aggregated per original clause) ----------------------
        val_metrics = _evaluate_aggregated(model, tokenizer, val_texts, val_labels, device)
        val_f1      = val_metrics['f1']

        logger.info(f'Epoch {epoch}/{epochs} -- Loss: {avg_loss:.4f} | '
                    f'F1: {val_f1:.4f} | Prec: {val_metrics["precision"]:.4f} | '
                    f'Rec: {val_metrics["recall"]:.4f}')
        history.append({'epoch': epoch, 'loss': avg_loss, **val_metrics})

        # -- Checkpoint ----------------------------------------------------
        if val_f1 > best_f1:
            best_f1, best_epoch, no_improve = val_f1, epoch, 0
            best_dir.mkdir(parents=True, exist_ok=True)
            if BACKBONE_USE_MEAN_POOL.get(backbone, False):
                model.save_pretrained(str(best_dir))
            else:
                model.model.save_pretrained(str(best_dir))
            tokenizer.save_pretrained(str(best_dir))
            logger.info(f'  -> Best checkpoint saved (F1={val_f1:.4f})')
        else:
            no_improve += 1

        torch.save({'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict()}, str(state_pt))
        state_json.write_text(json.dumps({
            'epoch': epoch, 'best_f1': best_f1, 'best_epoch': best_epoch,
            'no_improve': no_improve, 'history': history
        }))

        if epoch >= min_epochs and no_improve >= patience:
            logger.info(f'Early stopping at epoch {epoch} (patience={patience}, min_epochs={min_epochs})')
            break

    logger.info(f'[{backbone}] Done. Best F1={best_f1:.4f} at epoch {best_epoch}')
    for fp in [state_json, state_pt]:
        if fp.exists():
            fp.unlink()
    return {'backbone': backbone, 'best_f1': best_f1,
            'best_epoch': best_epoch, 'history': history}

print('Trainer defined (fp16, ASL, sliding-window training, aggregated val evaluation).')

In [ ]:
def get_probs(backbone: str, texts: List[str]) -> np.ndarray:
    '''Load checkpoint and return sigmoid probs for all texts.'''
    device   = 'cuda' if torch.cuda.is_available() else 'cpu'
    ckpt_dir = BACKBONE_DIR_MAP[backbone] / 'best'
    use_fast = (backbone != 'legalbert')
    tok      = AutoTokenizer.from_pretrained(str(ckpt_dir), use_fast=use_fast)
    # Use the correct model class — MeanPool for DeBERTa, standard for Legal-BERT
    if BACKBONE_USE_MEAN_POOL.get(backbone, False):
        mdl = MeanPoolClassifierModel.from_pretrained(str(ckpt_dir)).to(device)
    else:
        mdl = ClauseClassifierModel(str(ckpt_dir)).to(device)
    mdl.eval()
    all_probs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), BATCH_SIZE * 2), desc=f'  {backbone} probs', leave=False):
            batch = texts[i : i + BATCH_SIZE * 2]
            enc   = tok(batch, max_length=MAX_LENGTH, padding=True, truncation=True, return_tensors='pt').to(device)
            # Both model classes return plain logit tensors (not objects with .logits)
            logits = mdl(enc['input_ids'], enc['attention_mask'])
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.vstack(all_probs)


def find_temperature(probs: np.ndarray, y_true: np.ndarray) -> float:
    '''Find temperature T in [0.5..2.0] that minimises val binary cross-entropy.

    Temperature scaling: calibrated_prob = sigmoid(logit(p) / T)
      T > 1  ->  softens probs (model was overconfident)
      T < 1  ->  sharpens probs (model was underconfident)
      T = 1  ->  no change
    '''
    import torch.nn.functional as F
    probs_clipped = np.clip(probs, 1e-7, 1 - 1e-7)
    logits_t = torch.tensor(np.log(probs_clipped / (1 - probs_clipped)), dtype=torch.float32)
    labels_t = torch.tensor(y_true, dtype=torch.float32)
    best_T, best_nll = 1.0, float('inf')
    for T in [round(t * 0.1, 1) for t in range(5, 21)]:  # 0.5 … 2.0
        nll = F.binary_cross_entropy_with_logits(logits_t / T, labels_t).item()
        if nll < best_nll:
            best_nll, best_T = nll, T
    return best_T


def calibrate(probs: np.ndarray, temperature: float) -> np.ndarray:
    '''Apply temperature scaling to a probability array.'''
    if temperature == 1.0:
        return probs
    clipped = np.clip(probs, 1e-7, 1 - 1e-7)
    logits  = np.log(clipped / (1 - clipped))
    return 1.0 / (1.0 + np.exp(-logits / temperature))


def optimise_thresholds(probs: np.ndarray, y_true: np.ndarray) -> np.ndarray:
    '''Find per-class threshold in [0.1..0.9] maximising per-class F1.
    Must be called on calibrated probs (after temperature scaling).
    '''
    thresholds = np.full(NUM_CLAUSE_TYPES, 0.5)
    for i in range(NUM_CLAUSE_TYPES):
        if y_true[:, i].sum() == 0:
            continue
        best = -1.0
        for t in [round(x * 0.1, 1) for x in range(1, 10)]:
            f1 = f1_score(y_true[:, i], (probs[:, i] >= t).astype(int), average='binary', zero_division=0)
            if f1 > best:
                best, thresholds[i] = f1, t
    return thresholds


def run_model_selection() -> Dict:
    '''Compare single models and ensemble on val set. Save production config.

    Correct order: temperature -> calibrated probs -> thresholds.
    Thresholds are found on calibrated probs because inference applies
    temperature before threshold comparison.
    '''
    val_texts, val_labels = load_split('val')
    y_true    = np.array(val_labels, dtype=np.float32)
    available = {}

    for bb in ('legalbert', 'deberta', 'legalroberta'):
        ckpt = BACKBONE_DIR_MAP[bb] / 'best'
        if ckpt.exists():
            logger.info(f'Loading {bb} for model selection...')
            available[bb] = get_probs(bb, val_texts)
        else:
            logger.info(f'No checkpoint for {bb} — skipping')

    if not available:
        raise RuntimeError('No checkpoints found. Train at least one backbone first.')

    results = {}
    for bb, probs in available.items():
        T         = find_temperature(probs, y_true)
        cal_probs = calibrate(probs, T)
        thr       = optimise_thresholds(cal_probs, y_true)
        f1        = f1_score(y_true, cal_probs >= thr, average='macro', zero_division=0)
        results[bb] = {
            'thresholds': thr, 'probs': probs, 'temperature': T, 'f1': f1,
            'w_lb': 1.0 if bb == 'legalbert' else 0.0,
            'w_db': 1.0 if bb == 'deberta' else 0.0,
            'w_lr': 1.0 if bb == 'legalroberta' else 0.0,
        }
        logger.info(f'  {bb} val F1 (T={T:.1f}, per-class thr): {f1:.4f}')

    weight_steps = [round(x * 0.1, 1) for x in range(0, 11)]

    def _eval_combo(pc):
        T   = find_temperature(pc, y_true)
        cal = calibrate(pc, T)
        thr = optimise_thresholds(cal, y_true)
        f1  = f1_score(y_true, cal >= thr, average='macro', zero_division=0)
        return f1, thr, T

    pairs = [
        ('legalbert', 'deberta',      'w_lb', 'w_db'),
        ('legalbert', 'legalroberta', 'w_lb', 'w_lr'),
        ('deberta',   'legalroberta', 'w_db', 'w_lr'),
    ]
    for bk_a, bk_b, fa, fb in pairs:
        if bk_a not in available or bk_b not in available:
            continue
        best_f1, best_wa = -1.0, 0.5
        for wa in weight_steps:
            wb = round(1.0 - wa, 1)
            f1, _, _ = _eval_combo(wa * available[bk_a] + wb * available[bk_b])
            if f1 > best_f1:
                best_f1, best_wa = f1, wa
        best_wb = round(1.0 - best_wa, 1)
        combined = best_wa * available[bk_a] + best_wb * available[bk_b]
        f1, thr, T = _eval_combo(combined)
        key = f'ensemble_{bk_a}_{bk_b}'
        results[key] = {
            'thresholds': thr, 'probs': combined, 'temperature': T, 'f1': f1,
            'w_lb': best_wa if bk_a == 'legalbert'    else (best_wb if bk_b == 'legalbert'    else 0.0),
            'w_db': best_wa if bk_a == 'deberta'      else (best_wb if bk_b == 'deberta'      else 0.0),
            'w_lr': best_wa if bk_a == 'legalroberta' else (best_wb if bk_b == 'legalroberta' else 0.0),
        }
        logger.info(f'  {key} ({fa}={best_wa:.1f}, {fb}={best_wb:.1f}) val F1: {f1:.4f}')

    if all(b in available for b in ('legalbert', 'deberta', 'legalroberta')):
        best_f1 = -1.0
        best_w3 = (0.34, 0.33, 0.33)
        for w_lb in weight_steps:
            for w_db in weight_steps:
                w_lr = round(1.0 - w_lb - w_db, 1)
                if w_lr < 0:
                    continue
                combined = w_lb * available['legalbert'] + w_db * available['deberta'] + w_lr * available['legalroberta']
                f1, _, _ = _eval_combo(combined)
                if f1 > best_f1:
                    best_f1 = f1
                    best_w3 = (w_lb, w_db, w_lr)
        w_lb, w_db, w_lr = best_w3
        combined = w_lb * available['legalbert'] + w_db * available['deberta'] + w_lr * available['legalroberta']
        f1, thr, T = _eval_combo(combined)
        results['ensemble'] = {
            'thresholds': thr, 'probs': combined, 'temperature': T, 'f1': f1,
            'w_lb': w_lb, 'w_db': w_db, 'w_lr': w_lr,
        }
        logger.info(f'  ensemble (w_lb={w_lb:.1f}, w_db={w_db:.1f}, w_lr={w_lr:.1f}) val F1: {f1:.4f}')

    winner_key = max(results, key=lambda k: results[k]['f1'])
    winner     = results[winner_key]
    logger.info(f'Winner: {winner_key} (val F1={winner["f1"]:.4f}, T={winner["temperature"]:.1f})')

    prod = {
        'model_type':        winner_key,
        'weight_legalbert':  winner['w_lb'],
        'weight_deberta':    winner['w_db'],
        'weight_legalroberta': winner.get('w_lr', 0.0),
        'thresholds':        {ct: float(winner['thresholds'][i]) for i, ct in enumerate(CUAD_CLAUSE_TYPES)},
        'temperature':       winner['temperature'],
        'unknown_threshold': UNKNOWN_THRESH,
        'val_f1_macro':      round(winner['f1'], 6),
    }
    with open(PRODUCTION_CONFIG_PATH, 'w') as f:
        json.dump(prod, f, indent=2)
    logger.info(f'Production config saved -> {PRODUCTION_CONFIG_PATH}')
    return prod

print('ModelSelector defined.')

## Step 6 — Train Legal-BERT

⏱ *~2–3 hours on T4 GPU for 10 epochs. Checkpoint auto-saved to Drive after each improvement.*

In [ ]:
results_lb = train_backbone('legalbert')
print(f'\nLegal-BERT training complete.')
print(f'  Best val F1 : {results_lb["best_f1"]:.4f}')
print(f'  Best epoch  : {results_lb["best_epoch"]}')
print(f'  Checkpoint  : {CLASSIFIER_LEGALBERT_DIR / "best"}')

## (Info) Auto-Resume

If Colab disconnects mid-training, just re-run the training cell (Step 6 or Step 7).

The trainer automatically detects training_state.json in the model directory and resumes from the next epoch -- optimizer state, scheduler state, best F1, and history are all restored. No manual action needed.

In [ ]:
# Auto-resume is now built into train_backbone().
# This cell is kept as a placeholder and does nothing.
print("Auto-resume is handled automatically by train_backbone()")

## Step 7 — Train DeBERTa-v3

⏱ *~3–4 hours on T4 GPU for 10 epochs. Checkpoint auto-saved to Drive after each improvement.*

In [ ]:
results_db = train_backbone('deberta')
print(f'\nDeBERTa training complete.')
print(f'  Best val F1 : {results_db["best_f1"]:.4f}')
print(f'  Best epoch  : {results_db["best_epoch"]}')
print(f'  Checkpoint  : {CLASSIFIER_DEBERTA_DIR / "best"}')

## Step 7b — Train Legal-RoBERTa-large

Trains `lexlms/legal-roberta-large` (355M params). Uses batch_size=8, grad_accum=4, gradient checkpointing to fit in Colab VRAM.
Early stopping: patience=6, min_epochs=10, max=90.
Skip this cell if you only want LegalBERT + DeBERTa.

In [ ]:
results_lr = train_backbone('legalroberta')
print(f'\nLegal-RoBERTa training complete.')
print(f'  Best val F1 : {results_lr["best_f1"]:.4f}')
print(f'  Best epoch  : {results_lr["best_epoch"]}')
print(f'  Checkpoint  : {CLASSIFIER_LEGALROBERTA_DIR / "best"}')


## Step 8 — Model Selection & Threshold Optimisation

In [ ]:
prod_config = run_model_selection()
print(f'\nModel selection complete.')
print(f'  Winner          : {prod_config["model_type"]}')
print(f'  Val F1 (macro)  : {prod_config["val_f1_macro"]:.4f}')
print(f'  Weight LegalBERT: {prod_config["weight_legalbert"]}')
print(f'  Weight DeBERTa  : {prod_config["weight_deberta"]}')

## Step 9 — Download Outputs

Downloads zipped checkpoints + production config to your local machine.

**After downloading, unzip and place as shown:**
```
contract-intelligence-platform/
└── models/
    └── clause_classifier/
        ├── legalbert/
        │   └── best/          <- unzip legalbert_best.zip contents here
        ├── deberta/
        │   └── best/          <- unzip deberta_best.zip contents here
        └── production_config.json
```

In [ ]:
import shutil
from google.colab import files

download_items = [
    (CLASSIFIER_LEGALBERT_DIR / 'best', '/content/legalbert_best'),
    (CLASSIFIER_DEBERTA_DIR        / 'best', '/content/deberta_best'),
    (CLASSIFIER_LEGALROBERTA_DIR   / 'best', '/content/legalroberta_best'),
]

for src_dir, zip_base in download_items:
    if src_dir.exists():
        zip_path = shutil.make_archive(zip_base, 'zip', root_dir=src_dir, base_dir='.')
        size_mb  = Path(zip_path).stat().st_size / 1_048_576
        print(f'Downloading {Path(zip_path).name} ({size_mb:.0f} MB)...')
        files.download(zip_path)
    else:
        print(f'Skipping {src_dir.name} — checkpoint not found')

if PRODUCTION_CONFIG_PATH.exists():
    print('Downloading production_config.json...')
    files.download(str(PRODUCTION_CONFIG_PATH))

print('\nAll downloads complete.')

## Training Summary

In [ ]:
for result in [results_lb, results_db]:
    bb = result['backbone']
    df = pd.DataFrame(result['history'])
    print(f'\n=== {bb} ===')
    print(df[['epoch', 'loss', 'f1', 'precision', 'recall']].to_string(index=False))
    print(f'Best F1: {result["best_f1"]:.4f} at epoch {result["best_epoch"]}')

print(f'\n=== Production Config ===')
print(f'Winner : {prod_config["model_type"]}')
print(f'Val F1 : {prod_config["val_f1_macro"]:.4f}')